# Human-in-the-Loop: Interrupt Mechanics

This notebook covers the core LangGraph mechanics for pausing a graph and resuming it with human input:

- The `interrupt()` function (runtime interrupts)
- Compile-time `interrupt_before` / `interrupt_after`
- The resume lifecycle (what re-runs vs. what doesn't)
- Validating human input inside a single node via repeated `interrupt()` calls

These are the primitives that the companion **`02_HITL_Patterns.ipynb`** notebook builds into applied, LLM-backed workflows (approve/reject, edit/review, tool-call review).

## Method 1: Runtime Interrupts Using `interrupt()`

### Step 1: Import Required Libraries

In [ ]:
from typing import TypedDict
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command

: 

### Step 2: Define the State Schema

In [ ]:
class State(TypedDict):
    some_text: str

### Step 3: Create the Human Node Function

In [ ]:
def human_node(state: State):
    print(f"Node started with state: {state}")
    # Present current state to human and pause execution
    value = interrupt({
        "text_to_revise": state["some_text"]
    })
    print(f"Received human input: {value}")
    # When resumed, this will contain the human's input
    return {
        "some_text": value
    }

### Step 4: Build the Graph Structure

In [ ]:
# Build the graph
graph_builder = StateGraph(State)
graph_builder.add_node("human_node", human_node)
graph_builder.add_edge(START, "human_node")

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

### Step 5: Initialize Execution

Invoking the graph runs until it hits `interrupt()` inside `human_node`, then pauses.

In [ ]:
# Execute with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

result = graph.invoke({"some_text": "original text"}, config=config)

In [ ]:
print("Graph paused! Here's what happened:")
print(f"Current state: {result}")
print(f"Interrupt details: {result.get('__interrupt__')}")

In [ ]:
# Check the current state
current_state = graph.get_state(config)
print(f"Current graph state: {current_state.values}")
print(f"Next node to execute: {current_state.next}")

In [ ]:
# result will contain '__interrupt__' with the interrupt information
interrupt_data = result.get('__interrupt__')[0].value
interrupt_data
# interrupt_data will be: {"text_to_revise": "original text"}

In [ ]:
print("Resuming with human input...")
final_result = graph.invoke(Command(resume="Edited text"), config=config)
print(f"Final result: {final_result}")

## Method 2: Compile-Time Interrupts Using `interrupt_before` and `interrupt_after`

Unlike `interrupt()`, which pauses from inside a node, `interrupt_before`/`interrupt_after` are declared when you `compile()` the graph — no code changes needed inside the node itself. We build one graph (`graph_builder`) and reuse it for both variants below, since the node logic and edges are identical.

### Shared Graph Definition

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
import uuid

class SimpleState(TypedDict):
    message: str
    processed: bool

def processing_node(state: SimpleState):
    print(f"Processing node received: {state}")
    return {
        "message": f"Processed: {state['message']}",
        "processed": True
    }

def final_node(state: SimpleState):
    print(f"Final node received: {state}")
    return {
        "message": f"Final: {state['message']}",
        "processed": True
    }

# Build graph (shared by both interrupt_before and interrupt_after examples below)
graph_builder = StateGraph(SimpleState)
graph_builder.add_node("processing_node", processing_node)
graph_builder.add_node("final_node", final_node)
graph_builder.add_edge(START, "processing_node")
graph_builder.add_edge("processing_node", "final_node")

### `interrupt_before`

In [ ]:
print("=== Testing interrupt_before ===")
graph_before = graph_builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["processing_node"]
)

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        graph_before.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Starting execution
result = graph_before.invoke({"message": "Hello", "processed": False}, config=config)
print(result)

In [ ]:
state = graph_before.get_state(config)
print(f"Graph state: {state.values}")
print(f"Next node: {state.next}")

In [ ]:
# Pass None as input to resume from checkpoint
final_result = graph_before.invoke(None, config=config)
print(f"Final result: {final_result}")

### `interrupt_after`

Same `graph_builder`, compiled with `interrupt_after` instead of `interrupt_before` — the node still runs, but the graph pauses right after it finishes.

In [ ]:
print("=== Testing interrupt_after ===")
graph_after = graph_builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_after=["processing_node"]
)

In [ ]:
display(
    Image(
        graph_after.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Initial execution
for chunk in graph_after.stream({"message": "Stream test", "processed": False}, config=config):
    print(f"Chunk: {chunk}")

In [ ]:
state = graph_after.get_state(config)
print(f"Graph state: {state.values}")
print(f"Next node: {state.next}")

In [ ]:
print("Resuming with stream...")
# Resume execution
for chunk in graph_after.stream(None, config=config):
    print(f"Resume chunk: {chunk}")

## Resume Lifecycle

A key mechanic to understand: when a graph resumes after `interrupt()`, the **entire node re-runs from the top** — including any code before the `interrupt()` call. Only side effects outside the graph (like `print`) reveal this; the state itself only reflects what happens after the interrupt is satisfied.

In [ ]:
import uuid
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START
from langgraph.types import interrupt, Command


def demo_resume_lifecycle():
    class SimpleState(TypedDict):
        counter: int
        message: str

    def counting_node(state: SimpleState):
        print(f"Node execution started - Counter: {state['counter']}")

        # Simulate some work before interrupt
        new_counter = state['counter'] + 1
        print(f"Incremented counter to: {new_counter}")

        # Interrupt and ask for human input
        human_input = interrupt({
            "current_counter": new_counter,
            "prompt": "What message should I add?"
        })

        print(f"Received human input: {human_input}")

        return {
            "counter": new_counter,
            "message": human_input
        }

    # Build and compile graph
    graph_builder = StateGraph(SimpleState)
    graph_builder.add_node("counting_node", counting_node)
    graph_builder.add_edge(START, "counting_node")

    graph = graph_builder.compile(checkpointer=InMemorySaver())
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}

    print("=== Initial Execution ===")
    result = graph.invoke({"counter": 0, "message": ""}, config=config)

    print(f"Graph paused with state: {result}")
    print(f"Interrupt data: {result.get('__interrupt__')[0].value}")

    print("\n=== Resume Execution ===")
    final_result = graph.invoke(Command(resume="Hello from human!"), config=config)
    print(f"Final state: {final_result}")

# Run the demo — note "Node execution started" and "Incremented counter to" print TWICE:
# once on the initial run, once again when the node re-runs from the top on resume.
demo_resume_lifecycle()

## Validating Human Input Within a Node

If you need to validate the input provided by the human *inside the graph itself* (rather than on the client side), you can achieve this with multiple `interrupt()` calls in a loop, within a single node — each invalid attempt re-prompts with a new message until valid input is received.

In [ ]:
from typing import TypedDict
import uuid

from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
# Define graph state
class State(TypedDict):
    age: int

# Node that asks for human input and validates it
def get_valid_age(state: State) -> State:
    prompt = "Please enter your age (must be a non-negative integer)."

    while True:
        user_input = interrupt(prompt)

        # Validate the input
        try:
            age = int(user_input)
            if age < 0:
                raise ValueError("Age must be non-negative.")
            break  # Valid input received
        except (ValueError, TypeError):
            prompt = f"'{user_input}' is not valid. Please enter a non-negative integer for age."

    return {"age": age}

# Node that uses the valid input
def report_age(state: State) -> State:
    print(f"Human is {state['age']} years old.")
    return state

# Build the graph
builder = StateGraph(State)
builder.add_node("get_valid_age", get_valid_age)
builder.add_node("report_age", report_age)

builder.set_entry_point("get_valid_age")
builder.add_edge("get_valid_age", "report_age")
builder.add_edge("report_age", END)

# Create the graph with a memory checkpointer
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

In [ ]:
# Run the graph until the first interrupt
config = {"configurable": {"thread_id": uuid.uuid4()}}
result = graph.invoke({}, config=config)
print(result["__interrupt__"])  # First prompt: "Please enter your age..."

In [ ]:
# Simulate an invalid input (e.g., string instead of integer)
result = graph.invoke(Command(resume="not a number"), config=config)
print(result["__interrupt__"])  # Follow-up prompt with validation message

In [ ]:
# Simulate a second invalid input (e.g., negative number)
result = graph.invoke(Command(resume="-10"), config=config)
print(result["__interrupt__"])  # Another retry

In [ ]:
# Provide valid input
final_result = graph.invoke(Command(resume="25"), config=config)
print(final_result)  # Should include the valid age

## Summary

- `interrupt()` pauses execution *from inside a node* and returns a value to present to a human; resuming re-runs the whole node from the top with `Command(resume=...)` supplying the return value of the `interrupt()` call.
- `interrupt_before` / `interrupt_after` pause the graph *between nodes*, declared at `compile()` time, with no code changes inside the node.
- On resume, everything in the interrupted node before the `interrupt()` call re-executes — plan for idempotency or side-effect-free code before the pause point.
- A node can call `interrupt()` repeatedly in a loop to validate human input before proceeding.

Next: **`02_HITL_Patterns.ipynb`** applies these mechanics to real LLM-backed agents — approval gates, editable state, and reviewable tool calls.